In [1]:
# Our markdown document collection

import pathlib

compass_documents_path = pathlib.Path('compass_documents')

compass_documents = list(compass_documents_path.glob('*.md'))

compass_documents[0]

WindowsPath('compass_documents/annual_leave_planning.md')

In [2]:
# Let's sample some chunking


from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)


def make_to_text(md_header_split :str) -> str:
    chunk = ''
    some_title =  md_header_split.metadata.get("Header 1")
    some_section = md_header_split.metadata.get("Header 2")
    some_subsection = md_header_split.metadata.get("Header 3")

    if some_title :
        chunk = f"This text talks about {some_title}"
    if some_section :
        chunk = f"{chunk}. It is specific on {some_section}"
    if some_subsection :
        chunk = f"{chunk}. and {some_section}."
    else:
        chunk = f"{chunk}."
    
    return f"{chunk} Details follow {md_header_split.page_content}"

# Chunk a Markdown document
with compass_documents[0].open('r', encoding='utf-8') as f:
    markdown_document = f.read()

md_header_splits = markdown_splitter.split_text(markdown_document)


for md_header_split in md_header_splits:

    chunk = make_to_text(md_header_split)

    print(chunk)

c:\work\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


. Details follow ---
title: Annual Leave Planning
category: HR
topic: annual_leave_planning
language: en
source: internal_policy
doc_type: guide
---
This text talks about Annual Leave Planning. Details follow As annual leave must be administered by the end of March of the following calendar year, it is very important to plan vacation days as early as possible in order to avoid conflict with your workflow.
This text talks about Annual Leave Planning. It is specific on Planning Process. Details follow - Annual Leave Planning is carried out by submitting leave and maintaining it in a "draft" status.
- "Draft" leave is editable in order to make any changes required. When the dates have been finalised (always upon agreement with your team and/or customer), the standard leave request procedure must be followed.
- Once the relevant absence form has been sent to `absences@agileactors.com`, your annual leave submission status will be changed to "Approved".
This text talks about Annual Leave Pla

In [3]:
#Set up our client

import typesense

api_key = 'xyz'
host = 'localhost'
port = '8108'
protocol = 'http'

client = typesense.Client(
            {
                "nodes": [{"host": host, "port": port, "protocol": protocol}],
                "api_key": api_key,
                "connection_timeout_seconds": 60,
            }
        )

db_collections = client.collections.retrieve()

all_collections = [db_collection['name'] for db_collection in db_collections]
all_collections

['vectordb_tutorial']

In [4]:
# Set up our collection

if 'vectordb_tutorial' in all_collections:
  client.collections['vectordb_tutorial'].documents.delete({'truncate': True})
else:
  schema = {
    'name': 'vectordb_tutorial',
    'fields': [
      {
        'name'  :  'name_of_document',
        'type'  :  'string'
      },
      {
        'name'  :  'document_chunk',
        'type'  :  'string'
      },
      {
        "name" : "embedding",
        "type" : "float[]",
        "embed": {
          "from": [
            "document_chunk",
          ],
          "model_config": {
            "model_name": "ts/nomic-embed-text-v1.5"
          }
        }
      }
    ]
  }

  client.collections.create(schema)

In [5]:
# Let's index our documents

import tqdm

for compass_document in compass_documents:
    with compass_document.open("r", encoding="utf-8") as f:
        markdown_document = f.read()

        md_header_splits = markdown_splitter.split_text(markdown_document)
        print(f"Processing document: {compass_document.name}")
        for i in tqdm.tqdm(range(len(md_header_splits))):
            chunk = make_to_text(md_header_splits[i])

            document = {
                "name_of_document": compass_document.name,
                "document_chunk": chunk,
            }

            client.collections["vectordb_tutorial"].documents.create(document)

Processing document: annual_leave_planning.md


100%|██████████| 4/4 [00:00<00:00, 11.33it/s]


Processing document: business-travel-policy.md


100%|██████████| 9/9 [00:00<00:00, 12.09it/s]


Processing document: chapters.md


...

100%|██████████| 9/9 [00:01<00:00,  8.47it/s]


Processing document: timesheet_process.md


100%|██████████| 8/8 [00:00<00:00, 12.27it/s]


Processing document: work_premises_and_hours.md


100%|██████████| 6/6 [00:00<00:00, 10.84it/s]


In [6]:
search_parameters = {
'q'                          : 'Absences with payment',
'query_by'                   : 'embedding',
}

results = client.collections['vectordb_tutorial'].documents.search(search_parameters)
results

{'facet_counts': [],
 'found': 10,
 'hits': [{'document': {'document_chunk': 'This text talks about Paid Leave Categories. It is specific on Sick Leave (Up to 3 Days) - waiting period. Details follow - **Eligibility**: In the occasion that a professional is absent from work due to illness up to 3 days.\n- **Type**: Paid as the company fully compensates the waiting period.\n- **Required documents**: Absence form\n- **Duration**: for a period of up to 3 days\n- **Notes**:\nThe company trusts you! In line with our policy, the company has decided to take on the cost of illness and as such, the professional is compensated with a full working day’s wage up until 9 absent days per calendar year are accumulated. This applies specifically to individual (1 day)-2 or 3days and no more than 3 days in a row.  \nAny day exceeding 9 DAYS PER YEAR, the employee must provide the company with an EOPPY doctor’s certificate from the 1st day of illness. The procedure to be followed is found under sick leav

In [7]:
best_document_chunk = results['hits'][0]['document']['document_chunk']
best_document_chunk

'This text talks about Paid Leave Categories. It is specific on Sick Leave (Up to 3 Days) - waiting period. Details follow - **Eligibility**: In the occasion that a professional is absent from work due to illness up to 3 days.\n- **Type**: Paid as the company fully compensates the waiting period.\n- **Required documents**: Absence form\n- **Duration**: for a period of up to 3 days\n- **Notes**:\nThe company trusts you! In line with our policy, the company has decided to take on the cost of illness and as such, the professional is compensated with a full working day’s wage up until 9 absent days per calendar year are accumulated. This applies specifically to individual (1 day)-2 or 3days and no more than 3 days in a row.  \nAny day exceeding 9 DAYS PER YEAR, the employee must provide the company with an EOPPY doctor’s certificate from the 1st day of illness. The procedure to be followed is found under sick leave for a period of over 3 days.'

In [9]:

from langchain.chat_models import init_chat_model


my_model = init_chat_model(
    "LFM2.5-1.2B-Instruct-GGUF",
    configurable_fields="any",  # This allows us to configure other params like temperature, max_tokens, etc at runtime.
    model_provider="openai",
    temperature=0,
    base_url="http://localhost:8081/v1",
    api_key="blahblah"
)

ai_message = my_model.invoke(best_document_chunk)

ai_message.content

"This text outlines the details of **Paid Leave Categories**, specifically focusing on **Sick Leave**, which allows employees to take time off due to illness. Here's a breakdown of the key points:\n\n---\n\n### **Eligibility**\n- Applies when a professional is absent from work due to illness for up to **3 days**.\n\n---\n\n### **Type**\n- **Paid Leave**: The company fully compensates the employee during the absence.\n\n---\n\n### **Required Documents**\n- **Absence Form**: Must be submitted to the company.\n\n---\n\n### **Duration**\n- Up to **3 days** of leave per calendar year.\n\n---\n\n### **Notes**\n- **Company Policy**: The company covers the cost of the employee's absence during the leave.\n- **Compensation**: The employee is paid a full working day’s wage up to **9 absent days per calendar year**.\n- **Exceptions**:\n  - If the absence exceeds **9 days**, the employee must provide an **EOPPY doctor’s certificate** from the first day of illness.\n  - For absences longer than **3

In [13]:
print(ai_message.content)

This text outlines the details of **Paid Leave Categories**, specifically focusing on **Sick Leave**, which allows employees to take time off due to illness. Here's a breakdown of the key points:

---

### **Eligibility**
- Applies when a professional is absent from work due to illness for up to **3 days**.

---

### **Type**
- **Paid Leave**: The company fully compensates the employee during the absence.

---

### **Required Documents**
- **Absence Form**: Must be submitted to the company.

---

### **Duration**
- Up to **3 days** of leave per calendar year.

---

### **Notes**
- **Company Policy**: The company covers the cost of the employee's absence during the leave.
- **Compensation**: The employee is paid a full working day’s wage up to **9 absent days per calendar year**.
- **Exceptions**:
  - If the absence exceeds **9 days**, the employee must provide an **EOPPY doctor’s certificate** from the first day of illness.
  - For absences longer than **3 days**, the procedure for ext